In [36]:
# Import Dependencies
import yfinance
import pandas_datareader as pdr
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [2]:
pdr.famafrench.get_available_datasets()

['F-F_Research_Data_Factors',
 'F-F_Research_Data_Factors_weekly',
 'F-F_Research_Data_Factors_daily',
 'F-F_Research_Data_5_Factors_2x3',
 'F-F_Research_Data_5_Factors_2x3_daily',
 'Portfolios_Formed_on_ME',
 'Portfolios_Formed_on_ME_Wout_Div',
 'Portfolios_Formed_on_ME_Daily',
 'Portfolios_Formed_on_BE-ME',
 'Portfolios_Formed_on_BE-ME_Wout_Div',
 'Portfolios_Formed_on_BE-ME_Daily',
 'Portfolios_Formed_on_OP',
 'Portfolios_Formed_on_OP_Wout_Div',
 'Portfolios_Formed_on_OP_Daily',
 'Portfolios_Formed_on_INV',
 'Portfolios_Formed_on_INV_Wout_Div',
 'Portfolios_Formed_on_INV_Daily',
 '6_Portfolios_2x3',
 '6_Portfolios_2x3_Wout_Div',
 '6_Portfolios_2x3_weekly',
 '6_Portfolios_2x3_daily',
 '25_Portfolios_5x5',
 '25_Portfolios_5x5_Wout_Div',
 '25_Portfolios_5x5_Daily',
 '100_Portfolios_10x10',
 '100_Portfolios_10x10_Wout_Div',
 '100_Portfolios_10x10_Daily',
 '6_Portfolios_ME_OP_2x3',
 '6_Portfolios_ME_OP_2x3_Wout_Div',
 '6_Portfolios_ME_OP_2x3_daily',
 '25_Portfolios_ME_OP_5x5',
 '25_Portf

In [3]:
# Daily 3 Factor
df = pdr.famafrench.FamaFrenchReader('F-F_Research_Data_Factors_daily')

In [4]:
data = df.read()

/var/folders/mr/nzdp38cj3cz3b0gcgyvws4c40000gp/T/ipykernel_27912/1535062722.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  data = df.read()


In [5]:
data

{0:             Mkt-RF   SMB   HML    RF
 Date                                
 2021-04-12   -0.04 -0.60  0.56  0.00
 2021-04-13    0.41 -0.30 -1.60  0.00
 2021-04-14   -0.34  0.79  1.41  0.00
 2021-04-15    1.06 -0.44 -1.19  0.00
 2021-04-16    0.27 -0.36  0.65  0.00
 ...            ...   ...   ...   ...
 2026-02-23   -1.18 -0.30 -1.31  0.01
 2026-02-24    0.83  0.51 -0.66  0.01
 2026-02-25    0.79 -0.37  0.49  0.01
 2026-02-26   -0.47  0.63  0.32  0.01
 2026-02-27   -0.51 -0.44 -1.25  0.01
 
 [1227 rows x 4 columns],
 'DESCR': 'F-F Research Data Factors daily\n-------------------------------\n\nThis file was created by using the 202602 CRSP database. The Tbill return is the simple daily rate that, over the number of trading days compounds to 1-month TBill rate. The 1-month TBill rate data until 202405 are from Ibbotson Associates. Starting from 202406, the 1-month TBill rate is from ICE BofA US 1-Month Treasury Bill Index. Copyright 2026 Eugene F. Fama and Kenneth R. French\n\n  0 : 

In [6]:
data[0]

,Mkt-RF,SMB,HML,RF
Date,,,,
2021-04-12,-0.04,-0.60,0.56,0.00
2021-04-13,0.41,-0.30,-1.60,0.00
2021-04-14,-0.34,0.79,1.41,0.00
2021-04-15,1.06,-0.44,-1.19,0.00
2021-04-16,0.27,-0.36,0.65,0.00
...,...,...,...,...
2026-02-23,-1.18,-0.30,-1.31,0.01
2026-02-24,0.83,0.51,-0.66,0.01
2026-02-25,0.79,-0.37,0.49,0.01


In [7]:
AAPL = yfinance.Ticker("NVDA")

In [8]:
AAPL.info

{'address1': '2788 San Tomas Expressway',
 'city': 'Santa Clara',
 'state': 'CA',
 'zip': '95051',
 'country': 'United States',
 'phone': '408 486 2000',
 'website': 'https://www.nvidia.com',
 'industry': 'Semiconductors',
 'industryKey': 'semiconductors',
 'industryDisp': 'Semiconductors',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization

In [9]:
history = AAPL.history(period='5y')

In [10]:
history = history.reset_index()

In [11]:
history['Date'] = history['Date'].dt.date

In [12]:
history

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2021-04-09,14.178263,14.371776,14.139362,14.363795,195172000,0.0,0.0
1,2021-04-12,14.253825,15.313904,14.103704,15.170764,869324000,0.0,0.0
2,2021-04-13,15.193205,15.660527,15.087721,15.640079,676212000,0.0,0.0
3,2021-04-14,15.585715,15.680975,15.189214,15.238590,385500000,0.0,0.0
4,2021-04-15,15.623121,16.173484,15.592199,16.096678,598480000,0.0,0.0
...,...,...,...,...,...,...,...,...
1251,2026-04-02,172.179993,177.490005,171.369995,177.389999,143143200,0.0,0.0
1252,2026-04-06,177.160004,177.789993,175.759995,177.639999,107564300,0.0,0.0
1253,2026-04-07,175.729996,178.229996,173.660004,178.100006,132534900,0.0,0.0
1254,2026-04-08,184.500000,185.259995,180.300003,182.080002,147293100,0.0,0.0


In [13]:
history['Previous Close'] = history['Close'].shift(1)

In [14]:
history['Percentage Change'] = ((history['Close'] - history['Previous Close']) / history['Previous Close']) * 100

In [15]:
history = history.dropna()

In [16]:
condensed_history = history[['Date', 'Percentage Change']]

In [17]:
condensed_history

,Date,Percentage Change
1,2021-04-12,5.618074
2,2021-04-13,3.093546
3,2021-04-14,-2.567048
4,2021-04-15,5.631017
5,2021-04-16,-1.392729
...,...,...
1251,2026-04-02,0.933143
1252,2026-04-06,0.140932
1253,2026-04-07,0.258954
1254,2026-04-08,2.234697


In [18]:
Fama_3_Factor = data[0].reset_index()

In [19]:
Fama_3_Factor['Date'] = Fama_3_Factor['Date'].dt.date

In [20]:
Fama_3_Factor

,Date,Mkt-RF,SMB,HML,RF
0,2021-04-12,-0.04,-0.60,0.56,0.00
1,2021-04-13,0.41,-0.30,-1.60,0.00
2,2021-04-14,-0.34,0.79,1.41,0.00
3,2021-04-15,1.06,-0.44,-1.19,0.00
4,2021-04-16,0.27,-0.36,0.65,0.00
...,...,...,...,...,...
1222,2026-02-23,-1.18,-0.30,-1.31,0.01
1223,2026-02-24,0.83,0.51,-0.66,0.01
1224,2026-02-25,0.79,-0.37,0.49,0.01
1225,2026-02-26,-0.47,0.63,0.32,0.01


In [21]:
Combined = Fama_3_Factor.merge(condensed_history, on='Date', how='inner')

In [22]:
Combined

,Date,Mkt-RF,SMB,HML,RF,Percentage Change
0,2021-04-12,-0.04,-0.60,0.56,0.00,5.618074
1,2021-04-13,0.41,-0.30,-1.60,0.00,3.093546
2,2021-04-14,-0.34,0.79,1.41,0.00,-2.567048
3,2021-04-15,1.06,-0.44,-1.19,0.00,5.631017
4,2021-04-16,0.27,-0.36,0.65,0.00,-1.392729
...,...,...,...,...,...,...
1222,2026-02-23,-1.18,-0.30,-1.31,0.01,0.911389
1223,2026-02-24,0.83,0.51,-0.66,0.01,0.678672
1224,2026-02-25,0.79,-0.37,0.49,0.01,1.405230
1225,2026-02-26,-0.47,0.63,0.32,0.01,-5.456124


In [23]:
Combined['Percent Change minus RF'] = (Combined['Percentage Change'] - (Combined['RF']))


In [24]:
Combined = Combined.drop(columns = 'Percentage Change')

In [25]:
Combined.columns

Index(['Date', 'Mkt-RF', 'SMB', 'HML', 'RF', 'Percent Change minus RF'], dtype='object')

In [37]:
tscv = TimeSeriesSplit(n_splits=4, test_size=252, gap=5)

X = Combined[['Mkt-RF', 'SMB', 'HML']]
y = Combined[['Percent Change minus RF']]

for train_index, test_index in tscv.split(Combined):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Initialize and Fit
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred

    # Check your Factor Loadings (Betas) for this fold
    print(f"Mkt-RF Beta: {model.coef_[0]}")
    print(f"Alpha (Intercept): {model.intercept_}")
    print(f"R2 score: {model.score(X_test, y_test)}")
    # If the average residual is positive, the stock outperformed the model's expectations
    print(f"Mean Prediction Error: {residuals.mean()}")
    X_train_with_const = sm.add_constant(X_train)

    # 2. Fit the OLS (Ordinary Least Squares) model
    ols_model = sm.OLS(y_train, X_train_with_const).fit()

    # 3. Access the p-values
    p_values = ols_model.pvalues
    print(f"P-values for this fold:\n{p_values}")

    # 4. View the full academic summary
    print(ols_model.summary())

Mkt-RF Beta: [ 1.96497121 -0.18204476 -0.91088991]
Alpha (Intercept): [0.32176428]
R2 score: 0.7300948035833551
Mean Prediction Error: Percent Change minus RF   -0.233317
dtype: float64
P-values for this fold:
const     1.634735e-02
Mkt-RF    4.297249e-26
SMB       3.556057e-01
HML       2.455193e-12
dtype: float64
                               OLS Regression Results                              
Dep. Variable:     Percent Change minus RF   R-squared:                       0.575
Model:                                 OLS   Adj. R-squared:                  0.569
Method:                      Least Squares   F-statistic:                     94.72
Date:                     Thu, 09 Apr 2026   Prob (F-statistic):           8.37e-39
Time:                             14:03:17   Log-Likelihood:                -441.68
No. Observations:                      214   AIC:                             891.4
Df Residuals:                          210   BIC:                             904.8
Df Model:  